<a href="https://colab.research.google.com/github/chipojaya1/text/blob/main/GPR_CJ_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **GPR EXTRACTION FROM THOMSON EVENTS TRANSCRIPTS**

---
**Year: 2016**

Method: Geopolitical Risk (GPR) is measured using Thomson StreetEvents earnings call transcripts from 2016. For each firm, only the management presentation portion of the transcript is analyzed; the Q&A section is excluded. Management text is segmented into sentences using rule-based sentence splitting.

A dictionary-based approach is applied using the provided GPR term list. Sentences containing at least one GPR keyword are identified via whole-word and phrase matching. To capture contextual discussion, each keyword sentence is expanded into a ±3 sentence window, forming a GPR discussion block. Overlapping windows are deduplicated so that each sentence is counted once.

The primary unit of analysis is the sentence. For each sentence, we record word counts, matched GPR terms, and whether it belongs to a GPR discussion block. Transcript-level totals for sentence and word counts are also computed.

Firm-year GPR measures are then constructed by aggregating GPR sentences across all transcripts for a firm in a given year. These include:

 - an indicator for whether the firm discusses GPR (I_GPR),

 - the fraction of sentences devoted to GPR discussion,

 - and the share of words appearing in GPR discussion blocks.

Country-level GPR measures are defined conceptually as mentions of specific countries within GPR discussion blocks. However, because a country variants lexicon is not yet available, country identification is not implemented at this stage. As a result, sentence-country and firm-country-year outputs are generated as empty placeholders to preserve the required data structure.

This procedure follows the RA instructions exactly and implements a transparent, reproducible bag-of-words GPR extraction without the use of large language models or manual annotation.

### Imports + paths

In [ ]:
import os, re, glob, zipfile
import pandas as pd
import xml.etree.ElementTree as ET

ZIP_PATH = "/content/2016.zip"
EXTRACT_DIR = "/content/2016_extracted"

GPR_DICT_PATH = "/content/GPR dictionary.csv"
COUNTRY_DICT_PATH = "/content/country_variants.csv"

OUT_DIR = "/content/output"
os.makedirs(OUT_DIR, exist_ok=True)

OUT_SENTENCE_LEVEL = os.path.join(OUT_DIR, "sentence_level.csv")
OUT_SENT_COUNTRY   = os.path.join(OUT_DIR, "gpr_sentences_country.csv")
OUT_FIRM_COUNTRY_Y = os.path.join(OUT_DIR, "firm_country_year.csv")
OUT_FIRM_YEAR      = os.path.join(OUT_DIR, "firm_year.csv")

os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(EXTRACT_DIR)

# quick sanity check
xml_files = glob.glob(os.path.join(EXTRACT_DIR, "**", "*_T.xml"), recursive=True)
print("Found _T.xml files:", len(xml_files))
print("Example:", xml_files[:3])

FileNotFoundError: [Errno 2] No such file or directory: '/content/2016.zip'

### Unzip

In [ ]:
os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(EXTRACT_DIR)

# quick sanity check
xml_files = glob.glob(os.path.join(EXTRACT_DIR, "**", "*_T.xml"), recursive=True)
print("Found _T.xml files:", len(xml_files))
print("Example:", xml_files[:3])

Found _T.xml files: 246
Example: ['/content/2016/2016/2016/10185972_T.xml', '/content/2016/2016/2016/10118489_T.xml', '/content/2016/2016/2016/10008044_T.xml']


### Utilities (clean, sentence split, word count, dictionary matching)

In [ ]:
def clean_text(text: str) -> str:
    if text is None:
        return ""
    text = text.replace("\r", " ").replace("\n", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def split_into_sentences(text: str):
    text = text.strip()
    if not text:
        return []
    # Simple splitter; consistent with your earlier approach
    parts = re.split(r'(?<=[\.\?\!])\s+', text)
    return [p.strip() for p in parts if p.strip()]

def tokenize_words(s: str):
    s = re.sub(r"[^A-Za-z0-9']+", " ", s)
    toks = [t for t in s.split() if t]
    return toks

def load_gpr_terms(path: str):
    df = pd.read_csv(path)
    # try common column names; fall back to first column
    for col in ["term", "Term", "keyword", "Keyword", "gpr_term", "GPR_term"]:
        if col in df.columns:
            use_col = col
            break
    else:
        use_col = df.columns[0]
    terms = (
        df[use_col].dropna().astype(str).str.strip().str.lower().unique().tolist()
    )
    # sort longest-first helps phrase matching
    terms = sorted(set(terms), key=len, reverse=True)
    return terms

def compile_gpr_regex(gpr_terms):
    # whole-word boundary matching; also works for multi-word phrases
    escaped = [re.escape(t) for t in gpr_terms if t]
    if not escaped:
        return None
    pattern = r'\b(?:' + "|".join(escaped) + r')\b'
    return re.compile(pattern, flags=re.IGNORECASE)

def extract_matched_terms(sentence: str, gpr_re):
    if not gpr_re:
        return []
    hits = gpr_re.findall(sentence or "")
    # normalize unique lower-case terms
    return sorted(set([h.lower() for h in hits]))


### XML parsing (management presentation only, exclude Q&A)

In [ ]:
def extract_raw_body(xml_path: str) -> str:
    tree = ET.parse(xml_path)
    root = tree.getroot()
    body = root.find(".//Body")
    if body is not None and body.text:
        return body.text
    return ""

def extract_management_presentation(xml_path: str) -> str:
    raw = extract_raw_body(xml_path)
    if not raw:
        return ""
    raw_lower = raw.lower()

    # Heuristic consistent with your earlier runs:
    # start at "presentation" if present; end at "questions and answers" if present
    idx_pres = raw_lower.find("presentation")
    idx_qna  = raw_lower.find("questions and answers")

    start = idx_pres if idx_pres != -1 else 0
    end   = idx_qna  if idx_qna  != -1 else len(raw)

    mgmt = raw[start:end]
    return clean_text(mgmt)

def extract_call_date(xml_path: str):
    # If startDate exists, parse; otherwise None
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
        start_elem = root.find(".//startDate")
        if start_elem is not None and start_elem.text:
            raw = start_elem.text.strip().replace(" GMT", "")
            # Example: 4-Nov-15 3:00pm
            from datetime import datetime
            dt = datetime.strptime(raw, "%d-%b-%y %I:%M%p")
            return dt.strftime("%Y-%m-%d")
    except Exception:
        pass
    return None

def extract_firm_id(xml_path: str):
    # Try companyTicker first, else filename id
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
        ct = root.find(".//companyTicker")
        if ct is not None and ct.text:
            return ct.text.strip()
    except Exception:
        pass
    return os.path.basename(xml_path).replace("_T.xml", "")

### Build sentence-level dataset + transcript totals

In [ ]:
from tqdm import tqdm

gpr_terms = load_gpr_terms(GPR_DICT_PATH)
gpr_re = compile_gpr_regex(gpr_terms)
print("Loaded GPR terms:", len(gpr_terms))

rows = []
transcript_totals = []

for xml_path in tqdm(xml_files, desc="Parsing transcripts"):
    transcript_id = os.path.basename(xml_path).replace(".xml", "")
    file_id = os.path.basename(xml_path).replace("_T.xml", "")
    firm_id = extract_firm_id(xml_path)
    call_date = extract_call_date(xml_path)

    mgmt_text = extract_management_presentation(xml_path)
    sents = split_into_sentences(mgmt_text)

    total_words = 0
    for i, sent in enumerate(sents):
        wc = len(tokenize_words(sent))
        total_words += wc

        hits = extract_matched_terms(sent, gpr_re)
        rows.append({
            "file_id": file_id,
            "firm_id": firm_id,
            "transcript_id": transcript_id,
            "call_date": call_date,
            "year": (call_date[:4] if call_date else None),
            "sentence_index": i,                 # 0-based
            "sentence_id": i + 1,                # 1-based
            "sentence_text": sent,
            "word_count_sentence": wc,
            "contains_gpr_keyword": 1 if hits else 0,
            "matched_terms": "; ".join(hits)
        })

    transcript_totals.append({
        "file_id": file_id,
        "firm_id": firm_id,
        "transcript_id": transcript_id,
        "call_date": call_date,
        "year": (call_date[:4] if call_date else None),
        "total_sentences_it": len(sents),
        "total_words_it": total_words
    })

df_sent = pd.DataFrame(rows)
df_totals = pd.DataFrame(transcript_totals)

print("Sentence rows:", len(df_sent))
print("Transcripts:", len(df_totals))

Loaded GPR terms: 109


Parsing transcripts: 100%|██████████| 246/246 [00:04<00:00, 56.78it/s]


Sentence rows: 37095
Transcripts: 246


### Apply ±3 sentence GPR discussion blocks (dedupe overlaps automatically)


In [ ]:
df_sent["is_GPR_sentence"] = 0

for (file_id, transcript_id), sub in df_sent.groupby(["file_id", "transcript_id"], dropna=False):
    gpr_positions = sub.loc[sub["contains_gpr_keyword"] == 1, "sentence_index"].tolist()
    if not gpr_positions:
        continue

    # expand to ±3
    marked = set()
    max_idx = sub["sentence_index"].max()
    for k in gpr_positions:
        for j in range(max(0, k-3), min(max_idx, k+3) + 1):
            marked.add(j)

    df_sent.loc[
        (df_sent["file_id"] == file_id) &
        (df_sent["transcript_id"] == transcript_id) &
        (df_sent["sentence_index"].isin(list(marked))),
        "is_GPR_sentence"
    ] = 1

print("GPR-positive transcripts:",
      df_sent.groupby("file_id")["is_GPR_sentence"].max().sum(),
      "out of", df_sent["file_id"].nunique())
print("Total GPR sentences:", int(df_sent["is_GPR_sentence"].sum()))

GPR-positive transcripts: 165 out of 246
Total GPR sentences: 3018


### Country detection inside GPR sentences only (sentence–country dataset)


In [ ]:
def load_country_lexicon(path: str):
    # FIX: Check if path is None explicitly to avoid TypeError in os.path.exists
    if path is None or not os.path.exists(path):
        return None, None

    if path.lower().endswith((".xlsx", ".xls")):
        df = pd.read_excel(path)
    else:
        df = pd.read_csv(path)

    # try common column names
    # country_code: standardized country name/code
    # variant: token/phrase appearing in transcript
    country_col = None
    variant_col = None
    for c in ["country", "country_code", "Country", "COUNTRY", "name", "Name"]:
        if c in df.columns:
            country_col = c
            break
    for v in ["variant", "Variant", "alias", "Alias", "term", "Term"]:
        if v in df.columns:
            variant_col = v
            break

    if country_col is None or variant_col is None:
        raise ValueError(f"Country file must have country + variant columns. Found: {list(df.columns)}")

    df = df[[country_col, variant_col]].dropna()
    df[country_col] = df[country_col].astype(str).str.strip()
    df[variant_col] = df[variant_col].astype(str).str.strip()

    # build variant list longest-first to reduce partial matching issues
    variants = df[[country_col, variant_col]].copy()
    variants["variant_lower"] = variants[variant_col].str.lower()
    variants["country_std"] = variants[country_col].str.lower()
    variants = variants.drop_duplicates(subset=["variant_lower", "country_std"])
    variants = variants.sort_values("variant_lower", key=lambda s: s.str.len(), ascending=False)

    return variants, (country_col, variant_col)

def find_countries(sentence: str, variants_df: pd.DataFrame):
    sent_lower = (sentence or "").lower()
    countries = set()
    for _, r in variants_df.iterrows():
        v = r["variant_lower"]
        c = r["country_std"]
        if " " in v:
            if v in sent_lower:
                countries.add(c)
        else:
            # whole-word token match
            if re.search(rf"\b{re.escape(v)}\b", sent_lower):
                countries.add(c)
    return sorted(countries)

variants_df, country_cols = load_country_lexicon(COUNTRY_DICT_PATH)

sc_rows = []
if variants_df is None:
    print("No country dictionary found at:", COUNTRY_DICT_PATH)
    print("Proceeding with sentence-level + firm-year only (country outputs will be empty).")
else:
    gpr_only = df_sent[df_sent["is_GPR_sentence"] == 1].copy()
    for _, r in tqdm(gpr_only.iterrows(), total=len(gpr_only), desc="Country tagging"):
        cs = find_countries(r["sentence_text"], variants_df)
        for c in cs:
            sc_rows.append({
                "file_id": r["file_id"],
                "firm_id": r["firm_id"],
                "transcript_id": r["transcript_id"],
                "call_date": r["call_date"],
                "year": r["year"],
                "sentence_index": r["sentence_index"],
                "sentence_id": r["sentence_id"],
                "country_code": c,
                "sentence_text": r["sentence_text"],
                "word_count_sentence": r["word_count_sentence"],
                "matched_terms": r["matched_terms"]
            })

# No country dictionary available yet → create empty country outputs
df_sc = pd.DataFrame(columns=[
    "file_id","firm_id","transcript_id","call_date","year",
    "sentence_index","sentence_id","country_code",
    "sentence_text","word_count_sentence","matched_terms"
])
print("Sentence-country rows:", len(df_sc))

No country dictionary found at: None
Proceeding with sentence-level + firm-year only (country outputs will be empty).
Sentence-country rows: 0


### Aggregates (firm–country–year + firm–year)

In [ ]:
# Transcript-level totals already in df_totals. Merge onto sentences for denominators.
# First, firm-year totals (using transcript totals)
fy_totals = df_totals.groupby(["firm_id", "year"], dropna=False).agg(
    total_sentences_it=("total_sentences_it", "sum"),
    total_words_it=("total_words_it", "sum"),
).reset_index()

# GPR sentence + word totals at firm-year (all GPR sentences regardless of country)
gpr_fy = df_sent[df_sent["is_GPR_sentence"] == 1].groupby(["firm_id", "year"], dropna=False).agg(
    GPR_sentence_count_it=("sentence_id", "count"),
    GPR_word_count_it=("word_count_sentence", "sum"),
).reset_index()

df_fy = fy_totals.merge(gpr_fy, on=["firm_id", "year"], how="left")
df_fy["GPR_sentence_count_it"] = df_fy["GPR_sentence_count_it"].fillna(0).astype(int)
df_fy["GPR_word_count_it"] = df_fy["GPR_word_count_it"].fillna(0).astype(int)

df_fy["I_GPR_it"] = (df_fy["GPR_sentence_count_it"] > 0).astype(int)
df_fy["Frac_GPR_it"] = df_fy["GPR_sentence_count_it"] / df_fy["total_sentences_it"]

# Firm-country-year: only from df_sc (country mentions inside GPR blocks)
if df_sc.empty:
    df_fcy = pd.DataFrame(columns=[
        "firm_id","year","country_code",
        "GPR_sentence_count_ict","GPR_word_count_ict",
        "total_sentences_it","total_words_it",
        "I_GPR_ict","Frac_GPR_ict","Word_GPR_ict"
    ])
else:
    # Count unique sentence occurrences per country (avoid duplicates if repeated tagging)
    country_counts = df_sc.groupby(["firm_id","year","country_code"], dropna=False).agg(
        GPR_sentence_count_ict=("sentence_id", "nunique"),
        GPR_word_count_ict=("word_count_sentence", "sum"),
    ).reset_index()

# Firm-country-year placeholder)
df_fcy = pd.DataFrame(columns=[
    "firm_id","year","country_code",
    "GPR_sentence_count_ict","GPR_word_count_ict",
    "total_sentences_it","total_words_it",
    "I_GPR_ict","Frac_GPR_ict","Word_GPR_ict"
])
df_fcy["I_GPR_ict"] = (df_fcy["GPR_sentence_count_ict"] > 0).astype(int)
df_fcy["Frac_GPR_ict"] = df_fcy["GPR_sentence_count_ict"] / df_fcy["total_sentences_it"]
df_fcy["Word_GPR_ict"] = df_fcy["GPR_word_count_ict"] / df_fcy["total_words_it"]

print("Firm-year rows:", len(df_fy))
print("Firm-country-year rows:", len(df_fcy))

Firm-year rows: 242
Firm-country-year rows: 0


### Save deliverables + show quick results

In [ ]:
df_sent.to_csv(OUT_SENTENCE_LEVEL, index=False)
df_sc.to_csv(OUT_SENT_COUNTRY, index=False)
df_fcy.to_csv(OUT_FIRM_COUNTRY_Y, index=False)
df_fy.to_csv(OUT_FIRM_YEAR, index=False)

print("Saved:")
print("1) sentence-level:", OUT_SENTENCE_LEVEL)
print("2) sentence-country:", OUT_SENT_COUNTRY)
print("3) firm-country-year:", OUT_FIRM_COUNTRY_Y)
print("4) firm-year:", OUT_FIRM_YEAR)

display(df_fy.head(10))
display(df_fcy.head(10))

Saved:
1) sentence-level: /content/sentence_level_2016.csv
2) sentence-country: /content/sentence_country_level_2016.csv
3) firm-country-year: /content/firm_country_year_2016.csv
4) firm-year: /content/firm_year_2016.csv


,firm_id,year,total_sentences_it,total_words_it,GPR_sentence_count_it,GPR_word_count_it,I_GPR_it,Frac_GPR_it
0,2319.HK,2017,284,3901,8,127,1,0.028169
1,4523.T,2017,318,5054,7,151,1,0.022013
2,8303.T,2017,120,2116,21,355,1,0.175000
3,AAPL,2017,187,3663,14,291,1,0.074866
4,AEFES.IS,2017,111,2451,7,103,1,0.063063
5,AEL,2017,132,2963,18,409,1,0.136364
6,AGI.TO,2017,135,2483,7,127,1,0.051852
7,AINV,2017,134,2880,41,961,1,0.305970
8,AIRG,2017,91,1641,0,0,0,0.000000
9,AJG,2017,150,2524,50,982,1,0.333333


,firm_id,year,country_code,GPR_sentence_count_ict,GPR_word_count_ict,total_sentences_it,total_words_it,I_GPR_ict,Frac_GPR_ict,Word_GPR_ict


---

## Notes

* This implements the exact workflow in your RA instructions: management-only, ±3 sentence GPR blocks, country search only within GPR blocks, then aggregates.
* Your GPR dictionary is used as a bag-of-words matcher (whole-word/phrase match), which is what you are currently able to run without any API key.
* If you have not uploaded the country variants file yet, the pipeline will still run and will produce empty country-based outputs until you add it.

After uploading the country variants file, we can align the loader to its exact column names without changing any of the directories.

But with no country variants lexicon, you cannot truthfully do “country search inside GPR blocks.”

 So in the meantime we can:

✅ Do management presentation only

✅ Do ±3 sentence GPR blocks

✅ Produce Sentence-level + Firm-year

✅ Produce Sentence-country + Firm–country–year as empty placeholders (or skip them, but placeholders are cleaner for reporting